# **DSK807: Applied Machine Learning**
## Hands-on: From Single Models to Bagging and Boosting

In this short notebook we revisit two familiar datasets:

1. **Breast Cancer Wisconsin** (classification)
2. **Ames Housing**(regression)

In [ ]:
# Imports

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestClassifier,
    RandomForestRegressor,
    AdaBoostClassifier,
    AdaBoostRegressor,
    GradientBoostingClassifier,
    GradientBoostingRegressor
)

from xgboost import XGBClassifier, XGBRegressor


from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge, Lasso
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# **Part 1: Classification: Breast Cancer Wisconsin**

We use the same familiar classification dataset and a small subset of features.

**Task:** Predict whether a tumor belongs to the malignant or benign class and compare a single model, a single tree, bagging, and boosting methods.

In [2]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer(as_frame=True)

features = [
    "mean radius",
    "mean texture",
    "mean perimeter",
    "mean smoothness",
    "mean concavity"
]

X = data.frame[features]
y = data.frame["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Test set:    ", X_test.shape)
X.head()

Training set: (426, 5)
Test set:     (143, 5)


,mean radius,mean texture,mean perimeter,mean smoothness,mean concavity
0,17.99,10.38,122.80,0.11840,0.3001
1,20.57,17.77,132.90,0.08474,0.0869
2,19.69,21.25,130.00,0.10960,0.1974
3,11.42,20.38,77.58,0.14250,0.2414
4,20.29,14.34,135.10,0.10030,0.1980


In [3]:
RT = 42
from sklearn.metrics import precision_score, recall_score


In [4]:
classification_models = {
    "Logistic Regression": make_pipeline(
        MinMaxScaler(),
        LogisticRegression(max_iter=2000, random_state=RT)
    ),

    "Decision Tree": DecisionTreeClassifier(
        max_depth=6,
        random_state=RT
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=100,
        random_state=RT
    ),

    "AdaBoost": AdaBoostClassifier(
        n_estimators=100,
        learning_rate=0.1,
        random_state=RT
    ),

    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=RT
    ),

    "XGBoost": XGBClassifier(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=3,
        random_state=RT,
        eval_metric="logloss"
    )
}

classification_results = []

for name, model in classification_models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    classification_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),

        "F1": f1_score(y_test, pred)
    })



classification_results = (
    pd.DataFrame(classification_results)
    .sort_values("F1", ascending=False)
    .reset_index(drop=True)
)

classification_results.round(3)

,Model,Accuracy,Precision,Recall,F1
0,Decision Tree,0.951,0.956,0.967,0.961
1,XGBoost,0.951,0.956,0.967,0.961
2,Logistic Regression,0.944,0.927,0.989,0.957
3,Gradient Boosting,0.944,0.956,0.956,0.956
4,Random Forest,0.937,0.945,0.956,0.950
5,AdaBoost,0.937,0.955,0.944,0.950


# **Part 2: Regression: Ames House Prices**

We now repeat the comparison for regression using the familiar **Ames Housing** dataset.

**Task:** Predict `SalePrice` from a small set of numerical housing features and compare the same families of models.

In [8]:
from sklearn.datasets import fetch_openml

ames = fetch_openml(
    name="house_prices",
    as_frame=True,
    parser="auto"
)

housing = ames.frame.copy()

features = [
    "OverallQual",
    "GrLivArea",
    "GarageCars",
    "TotalBsmtSF",
    "FullBath",
    "YearBuilt",
    "YearRemodAdd",
    "1stFlrSF",
    "TotRmsAbvGrd"
]

X = housing[features]
y = housing["SalePrice"].astype(float)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42
)

imputer = SimpleImputer(strategy="median")

X_train = pd.DataFrame(
    imputer.fit_transform(X_train),
    columns=X.columns,
    index=X_train.index
)

X_test = pd.DataFrame(
    imputer.transform(X_test),
    columns=X.columns,
    index=X_test.index
)

print("Training set:", X_train.shape)
print("Test set:    ", X_test.shape)
X_train.head()

Training set: (1168, 9)
Test set:     (292, 9)


,OverallQual,GrLivArea,GarageCars,TotalBsmtSF,FullBath,YearBuilt,YearRemodAdd,1stFlrSF,TotRmsAbvGrd
254,5.0,1314.0,1.0,1314.0,1.0,1957.0,1957.0,1314.0,5.0
1066,6.0,1571.0,2.0,799.0,2.0,1993.0,1994.0,799.0,7.0
638,5.0,796.0,0.0,796.0,1.0,1910.0,1950.0,796.0,4.0
799,5.0,1768.0,1.0,731.0,1.0,1937.0,1950.0,981.0,7.0
380,5.0,1691.0,1.0,1026.0,2.0,1924.0,1950.0,1026.0,6.0


In [12]:
regression_models = {
    "Linear Regression": LinearRegression(),

    "Decision Tree": DecisionTreeRegressor(
        max_depth=5,
        random_state=42
    ),

    "Random Forest": RandomForestRegressor(
        n_estimators=100,
        random_state=42
    ),

    "AdaBoost": AdaBoostRegressor(
        n_estimators=100,
        learning_rate=0.1,
        random_state=42
    ),

    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=6,
        random_state=42
    ),

    "XGBoost": XGBRegressor(
        n_estimators=100,
        learning_rate=0.1,
        max_depth=6,
        random_state=42,
        objective="reg:squarederror"
    ),
    "Baseline (Mean)": DummyRegressor(strategy="mean"),

    "Ridge": make_pipeline(StandardScaler(), Ridge()),

    "Lasso": make_pipeline(StandardScaler(), Lasso(max_iter=10000)),

    "KNN": make_pipeline(StandardScaler(), KNeighborsRegressor()),
}





regression_results = []

for name, model in regression_models.items():
    model.fit(X_train, y_train)
    pred = model.predict(X_test)

    regression_results.append({
        "Model": name,
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
        "R²": r2_score(y_test, pred)
    })

regression_results = (
    pd.DataFrame(regression_results)
    .sort_values("RMSE")
    .reset_index(drop=True)
)

regression_results.round({
    "MAE": 0,
    "RMSE": 0,
    "R²": 3
})

,Model,MAE,RMSE,R²
0,XGBoost,18129.0,27385.0,0.902
1,Gradient Boosting,18702.0,28316.0,0.895
2,Random Forest,18865.0,28693.0,0.893
3,AdaBoost,24749.0,35276.0,0.838
4,KNN,21810.0,35573.0,0.835
5,Decision Tree,25256.0,35694.0,0.834
6,Linear Regression,24950.0,39513.0,0.796
7,Lasso,24950.0,39513.0,0.796
8,Ridge,24947.0,39518.0,0.796
9,Baseline (Mean),62576.0,87619.0,-0.001
